# Natural Language Processing for Automated Classification of Cleft and Craniofacial Procedures from Operative Notes


In [ ]:
import numpy as np
import pandas as pd
import openai
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier

from sklearn.model_selection import train_test_split, KFold

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score,
    f1_score,
    hamming_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    precision_recall_curve,
    auc,
    multilabel_confusion_matrix
)

from sklearn.model_selection import GroupKFold

from scipy.sparse import hstack


## Preprocessing

For preprocessing, we load spacy and our data. Then we extract procedure type labels

<b> notes_procedure</b> are the procedural notes as well as indications and operative findings <br>
<b> procedures </b> are raw procedures labels, such as  

L1= CL repair, primary
L2 = CL repair, minor revision; scar revision or
L3 = CL repair, major revision; by recreation of
defect


A1 = ABG (alveolar bone grafting), primary
A2 = ABG (alveolar bone grafting), revision


In [ ]:
# load data
# notes_procedure = 
# procedures = 

In [ ]:
nlp = spacy.load("en_core_web_md")

In [ ]:
# create label columns for individual procedures

def get_label(procedures, startchar):
    """
    Generates a binary label based on the starting character of the type of procedure specified. 
    """
    label = procedures.apply(lambda row: row.astype(str).str.startswith(startchar).any(), axis=1)
    label = label.astype(int)
    return label

def preprocess_text(text):
    """
    Tokenizes text using spaCy, removes punctuation and stopwords
    """
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha]

    return tokens

In [ ]:
tokens_list = []

for note in notes_procedure:
    if not pd.isna(note):
        clean_tokens = preprocess_text(note)
        tokens_list.append(clean_tokens)
    else:
        tokens_list.append("")

text_notes = [" ".join(tokens) for tokens in tokens_list]

In [ ]:
# get all labels for different types of procedures we're interested in
is_lip = get_label(procedures, 'L')
is_abg = get_label(procedures, 'A')
is_palate = get_label(procedures, 'P')
is_vpi = get_label(procedures, 'V')
is_fistula = get_label(procedures, 'F')
is_maxillary = get_label(procedures, 'M')
is_rhinoplasty = get_label(procedures, 'R')
is_ear = get_label(procedures, 'E')
is_cranial = get_label(procedures, 'C')

labels_all = pd.concat([is_lip, is_abg, is_palate, is_vpi, is_fistula, is_maxillary, is_rhinoplasty, is_ear], axis=1)
labels_all.columns = ['L','A','P','V','F','M','R','E']

In [ ]:
# Convert oronasal fistula, orthognathic repositioning, and auditory into an "Other" category. 

labels_array = labels_all.values

keep_indices = [0, 1, 2, 3, 6]   # L, A, P, V, R
other_indices = [4, 5, 7]        # F, M, E  are Other

other_procedures = labels_array[:, other_indices].sum(axis=1, keepdims=True)
other_procedures = (other_procedures > 0).astype(int)

y_combined = np.hstack([labels_array[:, keep_indices], other_procedures])
label_classes_new = ['L', 'A', 'P', 'V', 'R', 'Other']

## Primary Classification

This is the code used for the primary classifier. The primary classifier categorizes operative notes into multiple procedure types simultaneously using a One-vs-Rest strategy with Random Forest. It handles the multi-label nature of craniofacial surgery, where multiple procedures are often performed during a single operation.

text_notes is a list of all notes <br>
labels_all are the labels, which is a pandas DataFrame (630 rows × 8 columns) of all class labels. 

In [4]:
# Import data here
# text_notes = 
# labels_all = 

In [ ]:
# Primary classification

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fpr_list = {label: [] for label in label_classes_new} 
tpr_list = {label: [] for label in label_classes_new} 
prec_list = {label: [] for label in label_classes_new} 
rec_list = {label: [] for label in label_classes_new} 
auc_list = {label: [] for label in label_classes_new} 

f1_micro_tuned = []
f1_macro_tuned = []
auc_scores = []
hamming_scores = []

gkf = GroupKFold(n_splits=5)
for train_idx, test_idx in gkf.split(text_notes, stratify_labels, groups=patient_ids):

    X_train_text, X_test_text = np.array(text_notes)[train_idx], np.array(text_notes)[test_idx]
    y_train, y_test = y_combined[train_idx], y_combined[test_idx]

    vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1,2))
    X_train = vectorizer.fit_transform(X_train_text)
    X_test  = vectorizer.transform(X_test_text)

    classifier = OneVsRestClassifier(RandomForestClassifier(random_state=42))
    classifier.fit(X_train, y_train)

    y_proba = classifier.predict_proba(X_test)
    y_pred_default = classifier.predict(X_test)

    # optimize threshold per class
    thresholds_opt = {}
    predictions_optimized = np.zeros_like(y_test)

    for i, label in enumerate(label_classes_new):
        class_true = y_test[:, i]
        class_score = y_proba[:, i]

        # maximize f1
        thresholds_to_try = np.linspace(0.1, 0.9, 81)
        best_f1 = 0
        best_thresh = 0.5

        for thresh in thresholds_to_try:
            class_pred = (class_score >= thresh).astype(int)
            f1 = f1_score(class_true, class_pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = thresh

        thresholds_opt[label] = best_thresh
        predictions_optimized[:, i] = (class_score >= best_thresh).astype(int)


        class_auc = roc_auc_score(class_true, class_score)
        auc_list[label].append(class_auc)

        precision, recall, _ = precision_recall_curve(class_true, class_score)
        fpr, tpr, _ = roc_curve(class_true, class_score)

        prec_list[label].append(precision)
        rec_list[label].append(recall)
        fpr_list[label].append(fpr)
        tpr_list[label].append(tpr)


    
    f1_micro_tuned.append(f1_score(y_test, predictions_optimized, average='micro'))
    f1_macro_tuned.append(f1_score(y_test, predictions_optimized, average='macro'))
    auc_scores.append(roc_auc_score(y_test, y_proba, average='macro'))
    hamming_scores.append(hamming_loss(y_test, predictions_optimized))


print("AUC: ", np.nanmean(auc_scores), "±", np.nanstd(auc_scores))
print("F1 micro: ", np.nanmean(f1_micro_tuned), "±", np.nanstd(f1_micro_tuned))
print("F1 macro: ", np.nanmean(f1_macro_tuned), "±", np.nanstd(f1_macro_tuned))
print("Hamming Loss: ", np.nanmean(hamming_scores), "±", np.nanstd(hamming_scores))

In [ ]:
# Train the final classifier on all data
vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_all = vectorizer.fit_transform(text_notes)

final_classifier = OneVsRestClassifier(
    RandomForestClassifier(
    )
)
final_classifier.fit(X_all, y_combined)

## Secondary Classification

For the secondary classifiers, we create variables containing lip and ABG subclassifications (primary vs revision). Then we generate synthetic notes for the minority subclass and use them to train a classifier with cross validation

In [ ]:
def make_synthetic_notes(prompt,num):
    """
    Generates synthetic notes using OpenAI's GPT-4o model.
    """
    
    synthetic_notes = []
    for i in range(num):
        chat_completion = openai.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="gpt-4o",
        )
        
        synthetic_notes.append(chat_completion.choices[0].message.content)

    return synthetic_notes


def find_specific_procedure(row, procedure_columns, procedure_mapping):
    """
    Searches for a specific procedure by checking if any of the procedure columns in the given row
    match a procedure in the predefined procedure_mapping keys.
    """
    for col in procedure_columns:
        if row[col] in procedure_mapping:
            return procedure_mapping[row[col]]
    return 0

In [ ]:
# Example of synthetic note prompt generation

prompt_l2 = """
You are a writing an operative note for a plastic surgeon specializing in cleft and craniofacial procedures.  
Generate a formal "PROCEDURE IN DETAIL" section for a secondary cleft lip repair (minor revision).
Use realistic surgical terminology appropriate for plastic surgery
Use the provided examples as stylistic reference. Vary wording and syntax while preserving appropriate clinical realism.

[Example 1]

[Example 2]

[Example 3]
""

In [ ]:
procedure_columns = ['procedure_1', 'procedure_2', 'procedure_3', 'procedure_4', 'procedure_5']

lip_procedure_mapping = {'L1': 1, 'L2': 2, 'L3': 3}
labels_lip_specific = procedures.apply(
    find_specific_procedure, 
    axis=1, 
    procedure_columns=procedure_columns, 
    procedure_mapping=lip_procedure_mapping
)

abg_procedure_mapping = {'A1': 1, 'A2': 2}
labels_abg_specific = procedures.apply(
    find_specific_procedure, 
    axis=1, 
    procedure_columns=procedure_columns, 
    procedure_mapping=abg_procedure_mapping
)


In [2]:
def secondary_classifier_binary(notes, labels, synth_notes=None, synth_labels=None,
                                 pos_label=1, n_splits=3):
    """
    Performs stratified k-fold cross-validation for a binary classifier.
    Trains a RandomForest on TF-IDF features,tunes the decision threshold to maximize F1 score, 
    and returns ROC and PR curve data along with aggregated performance metrics (AUC, AUPRC, F1, Hamming loss)
    """
    
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fpr_list = []
    tpr_list = []
    prec_list = []
    rec_list = []

    acc_tuned = [] 
    f1_tuned = []
    hamming_tuned = []
    auc_list = []
    auprc_list = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(notes, labels), 1):
        
        x_train, x_test = np.array(notes)[train_idx], np.array(notes)[test_idx]
        y_train, y_test = np.array(labels)[train_idx], np.array(labels)[test_idx]

        x_train_all = list(synth_notes) + list(x_train)
        y_train_all = np.concatenate([synth_labels, y_train])

        vect = TfidfVectorizer(max_features=500, ngram_range=(1, 2),
                               stop_words='english', min_df=2)
        X_train = vect.fit_transform(x_train_all)
        X_test = vect.transform(x_test)

        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, y_train_all)

        y_proba = model.predict_proba(X_test)[:, 1]  # positive class only

        # threshold tuning for best F1
        best_thresh = 0
        best_f1 = 0
        y_pred_tuned = (y_proba >= 0.5).astype(int)

        for t in np.arange(0.0, 1.0, 0.05):
            preds = (y_proba >= t).astype(int)
            f1 = f1_score(y_test, preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = t
                y_pred_tuned = preds.copy()

        acc_tuned.append(accuracy_score(y_test, y_pred_tuned))
        f1_tuned.append(f1_score(y_test, y_pred_tuned, zero_division=0))
        hamming_tuned.append(hamming_loss(y_test, y_pred_tuned))

        auc_list.append(roc_auc_score(y_test, y_proba))

        prec, rec, _ = precision_recall_curve(y_test, y_proba)
        auprc_list.append(auc(rec, prec))

        fpr, tpr, _ = roc_curve(y_test, y_proba)
        fpr_list.append(fpr)
        tpr_list.append(tpr)
        prec_list.append(prec)
        rec_list.append(rec)

    print("AUC: ", np.nanmean(auc_list), "±", np.nanstd(auc_list))
    print("AUPRC: ", np.nanmean(auprc_list), "±", np.nanstd(auprc_list))
    print("F1: ", np.nanmean(f1_tuned), "±", np.nanstd(f1_tuned))
    print("Hamming Loss: ", np.nanmean(hamming_tuned), "±", np.nanstd(hamming_tuned))

    return fpr_list, tpr_list, prec_list, rec_list


In [ ]:
# lip_only_notes are the lip notes
# lip_only_labels are the labels (1 or 2) for lip notes
# synthetic_l2_notes are synthetic notes for minor revision
# synthetic_l3_notes are synthetic notes for major revision

fpr_list, tpr_list, precision_list, recall_list = secondary_classifier_binary(
    notes=lip_only_notes,synth_notes=list(synthetic_l2_notes) + list(synthetic_l3_notes),
    synth_labels=np.concatenate([np.full(len(synthetic_l2_notes), 2),np.full(len(synthetic_l3_notes), 3)]),
    labels=np.array([0 if l==1 else 1 for l in lip_only_labels]),
    pos_label=1
)

In [ ]:
# abg_only_notes are the ABG notes
# abg_only_labels are the labels (1 or 2) for ABG notes
# synthetic_a2_notes are synthetic notes for revision

fpr_list, tpr_list, precision_list, recall_list = secondary_classifier_binary
    notes=abg_only_notes,
    labels=np.array([0 if l==1 else 1 for l in abg_only_labels]),
    synth_notes=list(synthetic_a2_notes),
    synth_labels=np.full(len(synthetic_a2_notes), 1),
    pos_label=1
)

## Tertiary Classification

This is where we predict technique used in lip, palate, and VPI procedures. 


In [ ]:
# lip_technique label = 
# palate_technique_label = 


In [ ]:
def tertiary_classification_multi(notes,labels,label_classes,n_splits=3):
    """
    Performs stratified k-fold cross-validation for a multiclass classifier.
    Trains a RandomForest on TF-IDF features,tunes the decision threshold to maximize F1 score, 
    and returns ROC and PR curve data along with aggregated performance metrics (AUC, AUPRC, F1, Hamming loss)
    """

    fpr_dict = {lab: [] for lab in label_classes}
    tpr_dict = {lab: [] for lab in label_classes}
    auc_per_class = {lab: [] for lab in label_classes}
    auprc_per_class = {lab: [] for lab in label_classes}
    prec_dict = {lab: [] for lab in label_classes}
    rec_dict = {lab: [] for lab in label_classes}

    auc_list = []
    auprc_list = []
    f1_micro_list = []
    f1_macro_list = []
    hamming_list = []
    
    
    classifier = OneVsRestClassifier(RandomForestClassifier(class_weight='balanced'))
    
    skf = StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=42
    )
    
    for train_idx, test_idx in skf.split(notes, labels):
        X_train_text, X_test_text = notes.iloc[train_idx],notes.iloc[test_idx] 
        y_train,y_test = labels.iloc[train_idx],labels.iloc[test_idx]
        
        vectorizer = TfidfVectorizer(max_features=500,ngram_range=(1, 2))
        
        X_train = vectorizer.fit_transform(X_train_text)
        X_test  = vectorizer.transform(X_test_text)
        
        classifier.fit(X_train, y_train)
        y_score = classifier.predict_proba(X_test)
        y_pred  = classifier.predict(X_test)
        
        y_test_bin = label_binarize(y_test, classes=list(range(len(label_classes))))
        
        auc_list.append(roc_auc_score(y_test_bin,y_score,average="macro",multi_class="ovr"))
        
        fold_auprcs = []
        for i in range(len(label_classes)):
            prec, rec, _ = precision_recall_curve(y_test_bin[:, i],y_score[:, i])
            fold_auprcs.append(auc(rec, prec))
        if fold_auprcs:
            auprc_list.append(np.mean(fold_auprcs))
        
        f1_micro_list.append(f1_score(y_test, y_pred, average="micro"))
        f1_macro_list.append(f1_score(y_test, y_pred, average="macro"))
        hamming_list.append(hamming_loss(y_test, y_pred))
        
        for i, lab in enumerate(label_classes):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i],y_score[:, i])
            auc_c = roc_auc_score(y_test_bin[:, i],y_score[:, i])
            prec, rec, _ = precision_recall_curve(y_test_bin[:, i],y_score[:, i])
            auprc_c = auc(rec, prec)  
            
            fpr_dict[lab].append(fpr)
            tpr_dict[lab].append(tpr)
            auc_per_class[lab].append(auc_c)
            auprc_per_class[lab].append(auprc_c)  
            prec_dict[lab].append(prec)
            rec_dict[lab].append(rec)

    print("AUC: ", np.nanmean(auc_list), "±", np.nanstd(auc_list))
    print("AUPRC: ", np.nanmean(auprc_list), "±", np.nanstd(auprc_list))
    print("F1 micro: ", np.nanmean(f1_micro_list), "±", np.nanstd(f1_micro_list))
    print("F1 macro: ", np.nanmean(f1_macro_list), "±", np.nanstd(f1_macro_list))
    print("Hamming Loss: ", np.nanmean(hamming_list), "±", np.nanstd(hamming_list))
        
    return fpr_dict,tpr_dict,auc_per_class,auprc_per_class,prec_dict,rec_dict

In [ ]:
def tertiary_classifier_binary(notes, labels, n_splits=3):
    """
    Performs stratified k-fold cross-validation for a binary classifier.
    Trains a RandomForest on TF-IDF features,tunes the decision threshold to maximize F1 score, 
    and returns ROC and PR curve data along with aggregated performance metrics (AUC, AUPRC, F1, Hamming loss)
    """
    
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fpr_list = []
    tpr_list = []
    prec_list = []
    rec_list = []

    acc_tuned = [] 
    f1_tuned = []
    hamming_tuned = []
    auc_list = []
    auprc_list = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(notes, labels), 1):
        
        x_train, x_test = np.array(notes)[train_idx], np.array(notes)[test_idx]
        y_train, y_test = np.array(labels)[train_idx], np.array(labels)[test_idx]

        x_train_all = list(synth_notes) + list(x_train)
        y_train_all = np.concatenate([synth_labels, y_train])

        vect = TfidfVectorizer(max_features=500, ngram_range=(1, 2),
                               stop_words='english', min_df=2)
        X_train = vect.fit_transform(x_train_all)
        X_test = vect.transform(x_test)

        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, y_train_all)

        y_proba = model.predict_proba(X_test)[:, 1]  # positive class only

        # threshold tuning for best F1
        best_thresh = 0
        best_f1 = 0
        y_pred_tuned = (y_proba >= 0.5).astype(int)

        for t in np.arange(0.0, 1.0, 0.05):
            preds = (y_proba >= t).astype(int)
            f1 = f1_score(y_test, preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = t
                y_pred_tuned = preds.copy()

        acc_tuned.append(accuracy_score(y_test, y_pred_tuned))
        f1_tuned.append(f1_score(y_test, y_pred_tuned, zero_division=0))
        hamming_tuned.append(hamming_loss(y_test, y_pred_tuned))

        auc_list.append(roc_auc_score(y_test, y_proba))

        prec, rec, _ = precision_recall_curve(y_test, y_proba)
        auprc_list.append(auc(rec, prec))

        fpr, tpr, _ = roc_curve(y_test, y_proba)
        fpr_list.append(fpr)
        tpr_list.append(tpr)
        prec_list.append(prec)
        rec_list.append(rec)

    print("AUC: ", np.nanmean(auc_list), "±", np.nanstd(auc_list))
    print("AUPRC: ", np.nanmean(auprc_list), "±", np.nanstd(auprc_list))
    print("F1: ", np.nanmean(f1_tuned), "±", np.nanstd(f1_tuned))
    print("Hamming Loss: ", np.nanmean(hamming_tuned), "±", np.nanstd(hamming_tuned))

    return fpr_list, tpr_list, prec_list, rec_list


In [ ]:
results_lip = tertiary_classification_multi(
    notes=lip_only_notes, # lip only notes
    labels=lip_technique_label, # technique labels
    label_classes=[0, 1, 2, 3]
)

In [ ]:
results_palate = tertiary_classification_multi(
    notes=palate_only_notes, # palate only notes
    labels=palate_technique_label, # technique labels
    label_classes=[0, 2, 3, 4, 5, 6]
)

In [ ]:
results_vpi = tertiary_classification_binary(
    notes=vpi_only_notes, # vpi only notes
    labels=palate_technique_label, # technique labels
)